# Búsqueda de hiperparámetros (Optuna) para YOLOv8n

Notebook reproducible en **Google Colab** y también ejecutable en local.

Flujo:
1. Setup e imports
2. Carga del dataset
3. Definición del objetivo de Optuna (con pruning)
4. Ejecución de la búsqueda (por defecto 10 trials x 10 épocas)
5. Visualización en TensorBoard
6. Visualización de resultados de optimización (Optuna)
7. Carga del mejor `best.pt` y revisión de curvas/matriz
8. Inferencia con `kortxo.jpg` usando el mejor modelo

In [1]:
# Fase 0 (setup): instalación de dependencias + imports globales
# Breve: dejamos el entorno listo para entrenar YOLOv8 con Optuna + TensorBoard.

import sys

if 'google.colab' in sys.modules:
    !pip -q install ultralytics optuna tensorboard huggingface_hub pyyaml seaborn
else:
    !pip -q install ultralytics optuna tensorboard huggingface_hub pyyaml seaborn

from pathlib import Path

import matplotlib.pyplot as plt
import optuna
import pandas as pd
import seaborn as sns
import yaml
from huggingface_hub import snapshot_download
from IPython.display import Image, display
from optuna.visualization.matplotlib import plot_optimization_history, plot_param_importances
from ultralytics import YOLO

sns.set_theme(style='whitegrid')


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


/home/mikel/github/TKNIKA/kortxovision/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1) Carga del dataset
Breve: descargamos el dataset desde Hugging Face y localizamos su archivo `data.yml`.

In [2]:
DATASET_REPO = 'mikeldiez/kortxovision'
LOCAL_DATA_DIR = Path('./data/kortxovision').resolve()
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f'Descargando dataset {DATASET_REPO}...')
dataset_root = Path(
    snapshot_download(
        repo_id=DATASET_REPO,
        repo_type='dataset',
        local_dir=str(LOCAL_DATA_DIR),
        local_dir_use_symlinks=False,
    )
).resolve()
print('Dataset descargado en:', dataset_root)

candidate_yaml = (
    list(dataset_root.rglob('data.yml'))
    + list(dataset_root.rglob('data.yaml'))
    + list(dataset_root.rglob('dataset.yml'))
    + list(dataset_root.rglob('dataset.yaml'))
)

if not candidate_yaml:
    raise FileNotFoundError('No se encontró data.yml/data.yaml en el dataset descargado.')

data_yaml_path = candidate_yaml[0]
print('Usando YAML:', data_yaml_path)

with open(data_yaml_path, 'r', encoding='utf-8') as f:
    data_cfg = yaml.safe_load(f)

print('\nResumen data.yaml:')
print({'nc': data_cfg.get('nc'), 'train': data_cfg.get('train'), 'val': data_cfg.get('val')})

Descargando dataset mikeldiez/kortxovision...


/home/mikel/github/TKNIKA/kortxovision/.venv/lib/python3.12/site-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
Fetching ... files: 2102it [00:02, 803.72it/s]


Dataset descargado en: /home/mikel/github/TKNIKA/kortxovision/notebooks/data/kortxovision
Usando YAML: /home/mikel/github/TKNIKA/kortxovision/notebooks/data/kortxovision/data.yml

Resumen data.yaml:
{'nc': 17, 'train': 'images/train', 'val': 'images/val'}


## 2) Definir búsqueda de hiperparámetros
Breve: configuramos Optuna para maximizar `mAP50-95` de validación en entrenamientos cortos.

In [3]:
# Imports defensivos para Colab si se ejecuta esta celda aislada
from pathlib import Path
import optuna
import pandas as pd

# Configuración principal (editable para clase/demo)
MODEL_NAME = 'yolov8n.pt'
N_TRIALS = 10            # Número de experimentos (configurable)
EPOCHS_PER_TRIAL = 10    # Épocas por experimento (configurable)

IMG_SIZE = 640
BATCH_SIZE = 8
WORKERS = 2
DEVICE = None            # None = auto, o 'cpu' / '0'
SEED = 42

RUNS_DIR = Path('./runs/optuna').resolve()
RUNS_DIR.mkdir(parents=True, exist_ok=True)


def get_best_map5095(results_csv_path: Path) -> float:
    """Extrae el mejor mAP50-95(B) del CSV de resultados de Ultralytics."""
    df = pd.read_csv(results_csv_path)
    metric_col = 'metrics/mAP50-95(B)'
    if metric_col not in df.columns:
        raise KeyError(f'No se encontró la columna {metric_col} en {results_csv_path}')
    return float(df[metric_col].max())


def build_pruning_callback(trial: optuna.trial.Trial):
    """Callback para reportar mAP50-95 por época y permitir pruning real."""

    def _callback(trainer):
        metrics = getattr(trainer, 'metrics', None) or {}
        map50_95 = metrics.get('metrics/mAP50-95(B)', None)
        if map50_95 is None:
            return

        epoch = int(getattr(trainer, 'epoch', 0)) + 1
        trial.report(float(map50_95), step=epoch)

        if trial.should_prune():
            raise optuna.TrialPruned(f'Trial podado en época {epoch} con mAP50-95={map50_95:.4f}')

    return _callback


def objective(trial: optuna.trial.Trial) -> float:
    # Espacio de búsqueda (pequeño pero útil para formación)
    params = {
        'lr0': trial.suggest_float('lr0', 1e-4, 5e-2, log=True),
        'lrf': trial.suggest_float('lrf', 0.01, 0.5),
        'momentum': trial.suggest_float('momentum', 0.80, 0.98),
        'weight_decay': trial.suggest_float('weight_decay', 1e-6, 1e-2, log=True),
        'hsv_h': trial.suggest_float('hsv_h', 0.0, 0.05),
        'hsv_s': trial.suggest_float('hsv_s', 0.2, 0.9),
        'hsv_v': trial.suggest_float('hsv_v', 0.2, 0.9),
        'degrees': trial.suggest_float('degrees', 0.0, 10.0),
        'scale': trial.suggest_float('scale', 0.2, 0.9),
        'fliplr': trial.suggest_float('fliplr', 0.0, 0.5),
        'mosaic': trial.suggest_float('mosaic', 0.0, 1.0),
    }

    trial_name = f'trial_{trial.number:03d}'
    trial_dir = RUNS_DIR / trial_name
    model = YOLO(MODEL_NAME)
    model.add_callback('on_fit_epoch_end', build_pruning_callback(trial))

    try:
        model.train(
            data=str(data_yaml_path),
            epochs=EPOCHS_PER_TRIAL,
            imgsz=IMG_SIZE,
            batch=BATCH_SIZE,
            workers=WORKERS,
            device=DEVICE,
            seed=SEED,
            project=str(RUNS_DIR),
            name=trial_name,
            exist_ok=True,
            pretrained=True,
            deterministic=True,
            verbose=False,
            val=True,
            plots=True,
            **params,
        )
    except optuna.TrialPruned:
        trial.set_user_attr('trial_dir', str(trial_dir))
        raise

    results_csv = trial_dir / 'results.csv'
    best_model_path = trial_dir / 'weights' / 'best.pt'
    score = get_best_map5095(results_csv)

    # Guardamos metadatos del trial para usar luego en visualización/inferencia.
    trial.set_user_attr('best_map50_95', score)
    trial.set_user_attr('trial_dir', str(trial_dir))
    trial.set_user_attr('best_model_path', str(best_model_path))
    return score

## 3) Ejecutar Optuna
Breve: lanzamos los trials y construimos una tabla para comparar resultados.

In [4]:
study = optuna.create_study(
    study_name='yolov8n_kortxovision_optuna',
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=3, interval_steps=1),
)

study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

best_trial = study.best_trial
BEST_TRIAL_NUM = best_trial.number
BEST_TRIAL_DIR = Path(best_trial.user_attrs['trial_dir'])
BEST_MODEL_PATH = Path(best_trial.user_attrs['best_model_path'])

print('Mejor trial:', BEST_TRIAL_NUM)
print('Mejor mAP50-95:', round(best_trial.value, 4))
print('Ruta best.pt:', BEST_MODEL_PATH)
print('Mejores hiperparámetros:')
for k, v in best_trial.params.items():
    print(f'  - {k}: {v}')

trials_df = study.trials_dataframe(attrs=('number', 'value', 'params', 'state'))
trials_df = trials_df.sort_values('value', ascending=False).reset_index(drop=True)
trials_df.head(10)

[I 2026-04-23 10:34:53,738] A new study created in memory with name: yolov8n_kortxovision_optuna
  0%|          | 0/10 [00:00<?, ?it/s]

New https://pypi.org/project/ultralytics/8.4.41 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.203 🚀 Python-3.12.3 torch-2.8.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6141MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/mikel/github/TKNIKA/kortxovision/notebooks/data/kortxovision/data.yml, degrees=8.661761457749352, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.35403628889802274, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.007800932022121826, hsv_s=0.30919616423534185, hsv_v=0.24065852851773964, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0010253509690168502, lrf=

E0000 00:00:1776933295.231907    7577 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776933295.248298    7577 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776933295.306709    7577 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776933295.306767    7577 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776933295.306770    7577 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776933295.306772    7577 computation_placer.cc:177] computation placer already registered. Please check linka

TensorBoard: Start with 'tensorboard --logdir /home/mikel/github/TKNIKA/kortxovision/notebooks/runs/optuna/trial_000', view at http://localhost:6006/
Overriding model.yaml nc=80 with nc=17

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      7360  ultralytics.nn.modules.block.C2f             [32, 32, 1, True]             
  3                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  4                  -1  2     49664  ultralytics.nn.modules.block.C2f             [64, 64, 2, True]             
  5                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  6         

/home/mikel/github/TKNIKA/kortxovision/.venv/lib/python3.12/site-packages/optuna/trial/_trial.py:501: UserWarning: The reported value is ignored because this `step` 10 is already reported.
  warnings.warn(
Best trial: 0. Best value: 0.32043:  10%|█         | 1/10 [03:01<27:11, 181.24s/it]

...

Best trial: 2. Best value: 0.42496:  90%|█████████ | 9/10 [15:00<01:12, 72.76s/it]

[I 2026-04-23 10:49:54,174] Trial 8 pruned. Trial podado en época 4 con mAP50-95=0.1927
New https://pypi.org/project/ultralytics/8.4.41 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.203 🚀 Python-3.12.3 torch-2.8.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6141MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/mikel/github/TKNIKA/kortxovision/notebooks/data/kortxovision/data.yml, degrees=4.103829230356297, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.11439908274581123, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.02542853455823514, hsv_s=0.835296531748265, hsv_v=0.3745045604042124, imgsz=640, int8=Fal

Best trial: 2. Best value: 0.42496: 100%|██████████| 10/10 [15:57<00:00, 95.73s/it]

[I 2026-04-23 10:50:51,026] Trial 9 pruned. Trial podado en época 3 con mAP50-95=0.1513
Mejor trial: 2
Mejor mAP50-95: 0.425
Ruta best.pt: /home/mikel/github/TKNIKA/kortxovision/notebooks/runs/optuna/trial_002/weights/best.pt
Mejores hiperparámetros:
  - lr0: 0.0006144543785587475
  - lrf: 0.18951730321390894
  - momentum: 0.8820925971590665
  - weight_decay: 0.0013826232179369874
  - hsv_h: 0.009983689107917987
  - hsv_s: 0.5599641068895281
  - hsv_v: 0.6146901982034297
  - degrees: 0.46450412719997725
  - scale: 0.6252813963310069
  - fliplr: 0.08526206184364576
  - mosaic: 0.06505159298527952


,number,value,params_degrees,params_fliplr,params_hsv_h,params_hsv_s,params_hsv_v,params_lr0,params_lrf,params_momentum,params_mosaic,params_scale,params_weight_decay,state
0,2,0.42496,0.464504,0.085262,0.009984,0.559964,0.614690,0.000614,0.189517,0.882093,0.065052,0.625281,0.001383,COMPLETE
1,1,0.38885,4.319450,0.305926,0.009170,0.412970,0.567330,0.041472,0.417897,0.838221,0.139494,0.403860,0.000005,COMPLETE
2,0,0.32043,8.661761,0.354036,0.007801,0.309196,0.240659,0.001025,0.475850,0.931759,0.020584,0.620781,0.000248,COMPLETE
3,3,0.20837,1.220382,0.017194,0.004884,0.678963,0.508107,0.036393,0.483160,0.945512,0.909320,0.546624,0.000017,PRUNED
4,8,0.19266,4.937956,0.213771,0.038039,0.592894,0.739677,0.024806,0.241385,0.821527,0.025419,0.565913,0.000713,PRUNED
5,9,0.15133,4.103829,0.114399,0.025429,0.835297,0.374505,0.000196,0.025400,0.914554,0.076980,0.728886,0.000018,PRUNED
6,7,0.14279,3.109823,0.364803,0.031165,0.431629,0.244491,0.000158,0.185648,0.820856,0.637557,0.427628,0.002834,PRUNED
7,5,0.13852,8.287375,0.140467,0.016267,0.472074,0.389944,0.030769,0.053361,0.835277,0.542696,0.449727,0.000002,PRUNED
8,6,0.09840,8.154614,0.364504,0.038612,0.339101,0.203865,0.000240,0.403077,0.813419,0.771270,0.694800,0.008862,PRUNED
9,4,0.09568,7.751328,0.447414,0.027336,0.329398,0.878709,0.000499,0.334636,0.856108,0.597900,0.857649,0.000120,PRUNED


## 5) Visualización en TensorBoard
Breve: abrimos TensorBoard para comparar los trials de entrenamiento (ideal en Colab).

In [ ]:
if 'google.colab' in sys.modules:
    ip = get_ipython()
    try:
        ip.run_line_magic('load_ext', 'tensorboard')
    except Exception:
        # Si ya está cargada la extensión, continuamos.
        pass

    print('Abriendo TensorBoard sobre:', RUNS_DIR)
    ip.run_line_magic('tensorboard', f'--logdir {RUNS_DIR}')
else:
    print('En local, lanza TensorBoard así:')
    print(f'tensorboard --logdir "{RUNS_DIR}" --port 6006')

## 6) Visualización de resultados (Optuna)
Breve: inspeccionamos convergencia, relevancia de hiperparámetros y dispersión por trial.

In [ ]:
# 6.1 Historial de optimización
ax1 = plot_optimization_history(study)
ax1.set_title('Optuna: historial de mAP50-95')
plt.show()

# 6.2 Importancia de hiperparámetros
ax2 = plot_param_importances(study)
ax2.set_title('Optuna: importancia de hiperparámetros')
plt.show()

# 6.3 Ranking simple por trial
plt.figure(figsize=(10, 4))
sns.barplot(data=trials_df.sort_values('number'), x='number', y='value', color='steelblue')
plt.title('mAP50-95 por trial')
plt.xlabel('Trial')
plt.ylabel('mAP50-95')
plt.tight_layout()
plt.show()

# 6.4 Mejor fila (tabla)
trials_df.head(1)